In [ ]:
%pip install git+https://github.com/databricks-industry-solutions/x12-edi-parser

#### Need to set CATALOG.SCHEMA.VOLUME to something you can write to.

In [ ]:
from ember import *
from ember.hls.healthcare import HealthcareManager as hm
import json, os
from pyspark.sql.functions import col, explode, concat, lit, to_json, expr
from ember.hls.mapinarrow_functions import from_edi_exploded, get_exploded_schema, flatten_edi

CATALOG = 'jltz83_test'
SCHEMA = 'default'
VOLUME = 'jltz83_test_volume'

df = spark.read.text("file:///" + os.getcwd() + "/../sampledata/837/*txt", wholetext = True)

df2 = df.withColumn("pk", col("_metadata.file_name")).select("pk", "value")
result_df = df2.mapInArrow(from_edi_exploded, schema=get_exploded_schema())

tmp = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_tmp/edi_json"

line = expr("""
  concat(
    left(to_json(struct(coalesce(pk, '') as pk)),
         length(to_json(struct(coalesce(pk, '') as pk))) - 1),
    ',"edi_json":', edi_json, '}'
  )
""")

result_df.select(line.alias("value")).write.mode("overwrite").text(tmp)

claims = (
    spark.read.json(tmp)
        .withColumn("FunctionalGroup", explode(col("edi_json.FunctionalGroup")))
        .withColumn("Transaction", explode(col("FunctionalGroup.Transactions")))
        .withColumn("Claim", explode(col("Transaction.Claims")))
        .selectExpr(
            "pk as filename",
            "`edi_json`.`EDI.control_number`",
            "`edi_json`.`EDI.date`",
            "`edi_json`.`EDI.recipient_qualifier_id`",
            "`edi_json`.`EDI.sender_qualifier_id`",
            "`edi_json`.`EDI.standard_version`",
            "`edi_json`.`EDI.time`",
            "FunctionalGroup.*",
            "Transaction.*",
            "Claim.*",
        )
        .drop("Transactions", "Claims")
)

claims.filter(col("claim_header.claim_id")=='TD-R192ICE00087').display()